In [1]:
!nvidia-smi

Sat May  9 08:05:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [22]:
%%writefile pagerank_cuda.cu

#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>

#define N 1000
#define MAX_ITER 100
#define DAMPING 0.85

__global__ void pagerank_kernel(
    int *adj,
    int *out_degree,
    double *rank,
    double *new_rank
) {

    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < N) {

        double sum = 0.0;

        for (int j = 0; j < N; j++) {

            if (adj[j * N + i] == 1) {
                sum += rank[j] / out_degree[j];
            }
        }

        new_rank[i] =
            (1.0 - DAMPING) / N +
            DAMPING * sum;
    }
}

int main() {

    int i, j, iter;

    int *h_adj;
    int *h_out_degree;

    double *h_rank;
    double *h_new_rank;

    h_adj = (int *)malloc(N * N * sizeof(int));
    h_out_degree = (int *)malloc(N * sizeof(int));

    h_rank = (double *)malloc(N * sizeof(double));
    h_new_rank = (double *)malloc(N * sizeof(double));

    srand(42);

    for (i = 0; i < N; i++) {

        h_out_degree[i] = 0;

        for (j = 0; j < N; j++) {

            h_adj[i * N + j] =
                (rand() % 100 < 5) ? 1 : 0;

            if (h_adj[i * N + j] == 1)
                h_out_degree[i]++;
        }

        if (h_out_degree[i] == 0)
            h_out_degree[i] = 1;
    }

    for (i = 0; i < N; i++) {
        h_rank[i] = 1.0 / N;
    }

    int *d_adj;
    int *d_out_degree;

    double *d_rank;
    double *d_new_rank;

    cudaMalloc((void**)&d_adj,
               N * N * sizeof(int));

    cudaMalloc((void**)&d_out_degree,
               N * sizeof(int));

    cudaMalloc((void**)&d_rank,
               N * sizeof(double));

    cudaMalloc((void**)&d_new_rank,
               N * sizeof(double));

    cudaMemcpy(d_adj,
               h_adj,
               N * N * sizeof(int),
               cudaMemcpyHostToDevice);

    cudaMemcpy(d_out_degree,
               h_out_degree,
               N * sizeof(int),
               cudaMemcpyHostToDevice);

    cudaMemcpy(d_rank,
               h_rank,
               N * sizeof(double),
               cudaMemcpyHostToDevice);

    cudaEvent_t start, stop;

    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    int threadsPerBlock = 256;

    int blocksPerGrid =
        (N + threadsPerBlock - 1) / threadsPerBlock;

    for (iter = 0; iter < MAX_ITER; iter++) {

        pagerank_kernel<<<blocksPerGrid,
                          threadsPerBlock>>>(
            d_adj,
            d_out_degree,
            d_rank,
            d_new_rank
        );

        cudaDeviceSynchronize();

        double *temp = d_rank;
        d_rank = d_new_rank;
        d_new_rank = temp;
    }

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float milliseconds = 0;

    cudaEventElapsedTime(&milliseconds,
                         start,
                         stop);

    cudaMemcpy(h_rank,
               d_rank,
               N * sizeof(double),
               cudaMemcpyDeviceToHost);

    printf("\n===== CUDA PAGERANK =====\n");

    printf("Threads Per Block : %d\n",
           threadsPerBlock);

    printf("Execution Time    : %f ms\n",
           milliseconds);

    printf("\nSample Values:\n");

    for (i = 0; i < 10; i++) {
        printf("Node %d : %.6f\n",
               i,
               h_rank[i]);
    }

    cudaFree(d_adj);
    cudaFree(d_out_degree);
    cudaFree(d_rank);
    cudaFree(d_new_rank);

    free(h_adj);
    free(h_out_degree);
    free(h_rank);
    free(h_new_rank);

    return 0;
}

Overwriting pagerank_cuda.cu


In [23]:
!nvcc pagerank_cuda.cu -o pagerank_cuda

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [26]:
!./pagerank_cuda


===== CUDA PAGERANK =====
Threads Per Block : 256
Execution Time    : 117.057411 ms

Sample Values:
Node 0 : 0.001198
Node 1 : 0.000864
Node 2 : 0.001281
Node 3 : 0.001117
Node 4 : 0.001143
Node 5 : 0.001031
Node 6 : 0.001324
Node 7 : 0.000955
Node 8 : 0.001021
Node 9 : 0.000986
